# 01 · ทำความสะอาดข้อมูลและคำนวณความเสี่ยงสินค้าขาดสต็อก

**เป้าหมาย:** บอกว่าสินค้าแต่ละตัว "พอขายอีกกี่วัน" และเสี่ยงขาดหรือไม่ เพื่อให้เจ้าหน้าที่จัดซื้อตัดสินใจสั่งเติมได้ทัน

**ข้อมูลที่ใช้**
- ของอาจารย์ (`data/raw/`): `orders.csv`, `order_lines.csv`, `products.csv`, `inventory_snapshot.csv` (สต็อกตอน 06:00)
- `data/raw/inventory_updates.json` = สต็อกที่อัปเดตตอน 11:40 (คัดลอกจาก `api/fixtures/inventory.json` ในชุดของอาจารย์)
- ข้อมูลที่ทีมสร้างเอง (`data/generated/suppliers.csv`): ผู้ขายแต่ละหมวด ส่งของกี่วัน สั่งขั้นต่ำกี่ชิ้น

**ผลลัพธ์**: `data/cleaned/stock_risk.csv` (ส่งให้ n8n), ไฟล์บันทึกการแก้ข้อมูล และไฟล์รายการที่กักไว้ให้คนตรวจ

> กติกา: ไม่แก้ไฟล์ใน `data/raw/` เลย ทุกการแก้ไขเกิดในตัวแปรใหม่ แล้วบันทึกไว้ว่าแก้อะไร

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, GEN, CLEAN = ROOT / "data/raw", ROOT / "data/generated", ROOT / "data/cleaned"
GEN.mkdir(parents=True, exist_ok=True); CLEAN.mkdir(parents=True, exist_ok=True)

orders      = pd.read_csv(RAW / "orders.csv")
order_lines = pd.read_csv(RAW / "order_lines.csv")
products    = pd.read_csv(RAW / "products.csv")
inventory   = pd.read_csv(RAW / "inventory_snapshot.csv")
updates     = pd.DataFrame(json.loads((RAW / "inventory_updates.json").read_text()).values())

quality_log = []   # บันทึกปัญหาคุณภาพข้อมูลทุกข้อ: เจออะไร แก้ยังไง กี่แถว
for name, df in [("orders", orders), ("order_lines", order_lines), ("products", products), ("inventory", inventory)]:
    print(f"{name:12s} {df.shape[0]:5d} แถว  คอลัมน์: {list(df.columns)}")

orders        1504 แถว  คอลัมน์: ['order_id', 'customer_id', 'order_date', 'promised_date', 'export_status', 'exported_at']
order_lines   1504 แถว  คอลัมน์: ['order_id', 'product_id', 'quantity']
products        82 แถว  คอลัมน์: ['product_id', 'product_name', 'category', 'unit_price_thb']
inventory       82 แถว  คอลัมน์: ['product_id', 'location_id', 'available_qty', 'as_of']


## 1. ตรวจ key ที่ใช้เชื่อมตาราง

ระบบนี้เชื่อมตารางด้วย `order_id` (ออร์เดอร์ ↔ รายการสินค้า) และ `product_id` (รายการสินค้า ↔ สินค้า ↔ สต็อก)
ต้องเช็กก่อนว่ามีแถวที่เชื่อมไม่ติดหรือไม่ ถ้ามี ยอดขายจะหายไปเงียบ ๆ

In [2]:
checks = {
    "order_lines ที่หา order ไม่เจอ": (~order_lines.order_id.isin(orders.order_id)).sum(),
    "order_lines ที่หาสินค้าไม่เจอ": (~order_lines.product_id.isin(products.product_id)).sum(),
    "สินค้าที่ไม่มีข้อมูลสต็อก": (~products.product_id.isin(inventory.product_id)).sum(),
    "order_id ซ้ำ": orders.order_id.duplicated().sum(),
}
pd.Series(checks, name="จำนวน")

order_lines ที่หา order ไม่เจอ    0
order_lines ที่หาสินค้าไม่เจอ     0
สินค้าที่ไม่มีข้อมูลสต็อก         0
order_id ซ้ำ                      0
Name: จำนวน, dtype: int64

ผลคือเชื่อมได้ครบทุกแถว ไม่มี key ซ้ำ จึงใช้ `order_id` และ `product_id` เชื่อมได้อย่างมั่นใจ

## 2. ปัญหาที่ 1: วันที่สั่งซื้อมีหลายรูปแบบ และบางตัวกำกวม

วันที่ส่วนใหญ่เป็น `ปี-เดือน-วัน` แต่มีบางแถวเป็น `วัน/เดือน/ปี`
- `28/07/2026` ตีความได้แบบเดียว (ไม่มีเดือนที่ 28) → แปลงได้
- `07/08/2026` อาจเป็น 7 ส.ค. หรือ 8 ก.ค. → **ห้ามเดา** กักไว้ให้คนตรวจ

ตัวช่วยตรวจ: `order_id` มีวันที่ฝังอยู่ เช่น `ORD-260728-1842` = 2026-07-28 ใช้เทียบเป็นหลักฐานประกอบ

In [3]:
def parse_order_date(text):
    """คืน (วันที่, สถานะ) โดยไม่เดาเมื่อกำกวม"""
    text = str(text)
    iso = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    if pd.notna(iso):
        return iso, "ok"
    if "/" in text:
        a, b, y = text.split("/")
        a, b = int(a), int(b)
        if a > 12:   # ส่วนแรกเกิน 12 ต้องเป็นวัน → ตีความได้แบบเดียว
            return pd.Timestamp(int(y), b, a), "fixed_unambiguous"
        if b > 12:
            return pd.Timestamp(int(y), a, b), "fixed_unambiguous"
        return pd.NaT, "ambiguous"
    return pd.NaT, "unparseable"

orders_clean = orders.copy()
orders_clean[["order_date_clean", "date_status"]] = orders_clean.order_date.apply(
    lambda t: pd.Series(parse_order_date(t)))
orders_clean["date_from_order_id"] = pd.to_datetime(
    orders_clean.order_id.str.extract(r"ORD-(\d{6})-")[0], format="%y%m%d")

before_after = orders_clean[orders_clean.date_status != "ok"][
    ["order_id", "order_date", "order_date_clean", "date_status", "date_from_order_id"]]
before_after

,order_id,order_date,order_date_clean,date_status,date_from_order_id
1500,ORD-260728-1842,28/07/2026,2026-07-28,fixed_unambiguous,2026-07-28
1503,ORD-260730-2201,07/08/2026,NaT,ambiguous,2026-07-30


In [4]:
quarantine = before_after[before_after.date_status == "ambiguous"].merge(order_lines, on="order_id")
quarantine["reason"] = "วันที่กำกวม: อ่านได้ทั้ง 7 ส.ค. และ 8 ก.ค. และไม่ตรงกับวันที่ใน order_id"
quarantine.to_csv(CLEAN / "quarantine_order_dates.csv", index=False)

quality_log += [
    {"issue": "วันที่รูปแบบ วัน/เดือน/ปี ที่ตีความได้แบบเดียว", "rows": int((orders_clean.date_status == "fixed_unambiguous").sum()),
     "action": "แปลงเป็น ปี-เดือน-วัน", "evidence": "ส่วนแรกของวันที่มากกว่า 12 จึงเป็นวันแน่นอน"},
    {"issue": "วันที่กำกวม (07/08/2026)", "rows": int((orders_clean.date_status == "ambiguous").sum()),
     "action": "ไม่นับในยอดขาย กักไว้ให้คนตรวจ (quarantine_order_dates.csv)", "evidence": "อ่านได้ 2 แบบ และไม่ตรงกับวันที่ใน order_id"},
]
quarantine

,order_id,order_date,order_date_clean,date_status,date_from_order_id,product_id,quantity,reason
0,ORD-260730-2201,07/08/2026,NaT,ambiguous,2026-07-30,SKU-0005,1,วันที่กำกวม: อ่านได้ทั้ง 7 ส.ค. และ 8 ก.ค. และ...


## 3. ปัญหาที่ 2: ชื่อหมวดสินค้าเขียนไม่ตรงกัน

หมวดสินค้าสำคัญกับระบบนี้ เพราะเวลาส่งของของผู้ขายผูกกับหมวด ถ้าชื่อหมวดเพี้ยน สินค้าตัวนั้นจะหาเวลาส่งของไม่เจอ

In [5]:
print("ก่อนแก้:", products.category.value_counts().to_dict())
CATEGORY_MAP = {"kitchen": "Kitchen", "bedding": "Bedding", "Bed room": "Bedding", "storage": "Storage"}
products_clean = products.copy()
products_clean["category"] = products_clean.category.replace(CATEGORY_MAP)
print("หลังแก้:", products_clean.category.value_counts().to_dict())

changed = products[products.category != products_clean.category]
quality_log.append({"issue": "ชื่อหมวดสินค้าไม่สม่ำเสมอ (ตัวพิมพ์เล็ก, 'Bed room')", "rows": len(changed),
                    "action": "แปลงเป็น 4 หมวดมาตรฐานตาม CATEGORY_MAP",
                    "evidence": "; ".join(f"{r.product_id}: {r.category}" for r in changed.itertuples())})

ก่อนแก้: {'Bedding': 22, 'Kitchen': 20, 'Bathroom': 17, 'Storage': 14, 'Bed room': 4, 'kitchen': 2, 'bedding': 2, 'storage': 1}
หลังแก้: {'Bedding': 28, 'Kitchen': 22, 'Bathroom': 17, 'Storage': 15}


## 4. ปัญหาที่ 3: ตัวเลขสต็อกจากสองแหล่ง เวลาต่างกัน

ไฟล์ snapshot ถ่ายตอน 06:00 แต่มีข้อมูลอัปเดตตอน 11:40 ของสินค้าบางตัว ถ้าใช้ตัวเลขเก่า ผลประเมินอาจผิด
กฎของเรา: **ใช้ค่าที่ใหม่ที่สุด และเก็บเวลาของข้อมูลไว้เสมอ** เพื่อให้หน้าเว็บแสดงได้ว่าตัวเลขนี้เป็นของเวลาไหน

In [6]:
stock = inventory[["product_id", "available_qty", "as_of"]].assign(stock_source="snapshot 06:00")
latest = updates[["product_id", "available_qty", "as_of"]].assign(stock_source="inventory update 11:40")
compare = stock.merge(latest, on="product_id", suffixes=("_snapshot", "_update"))
display(compare[["product_id", "available_qty_snapshot", "as_of_snapshot", "available_qty_update", "as_of_update"]])

stock_all = pd.concat([stock, latest])
stock_all["as_of"] = pd.to_datetime(stock_all.as_of)
stock_latest = stock_all.sort_values("as_of").groupby("product_id").tail(1)

quality_log.append({"issue": "สต็อกสองแหล่งเวลาไม่ตรงกัน", "rows": len(compare),
                    "action": "ใช้ค่าล่าสุด และเก็บ stock_as_of กับ stock_source",
                    "evidence": "; ".join(f"{r.product_id}: {r.available_qty_snapshot}→{r.available_qty_update}" for r in compare.itertuples())})

,product_id,available_qty_snapshot,as_of_snapshot,available_qty_update,as_of_update
0,SKU-KT-104,8,2026-07-31T06:00:00+07:00,3,2026-07-31T11:40:00+07:00
1,SKU-BD-220,0,2026-07-31T06:00:00+07:00,12,2026-07-31T11:40:00+07:00


## 5. ข้อมูลใหม่ที่ทีมสร้าง: ผู้ขายแต่ละหมวด

ชุดข้อมูลของอาจารย์ไม่มีว่าสั่งของแล้วกี่วันถึง ทีมจึงสร้างตารางผู้ขายขึ้นเอง **เป็นสมมติฐาน ไม่ใช่ข้อมูลจริงของบริษัท**
- `lead_time_days` = สั่งแล้วกี่วันของถึงคลัง
- `min_order_qty` = สั่งขั้นต่ำต่อครั้ง

In [7]:
suppliers = pd.DataFrame([
    {"category": "Bathroom", "supplier_id": "SUP-BATH", "supplier_name": "Demo Bath Supply",  "lead_time_days": 5,  "min_order_qty": 10},
    {"category": "Kitchen",  "supplier_id": "SUP-KIT",  "supplier_name": "Demo Kitchenware", "lead_time_days": 7,  "min_order_qty": 6},
    {"category": "Storage",  "supplier_id": "SUP-STO",  "supplier_name": "Demo Storage Co.", "lead_time_days": 7,  "min_order_qty": 12},
    {"category": "Bedding",  "supplier_id": "SUP-BED",  "supplier_name": "Demo Textile",     "lead_time_days": 10, "min_order_qty": 8},
])
suppliers.to_csv(GEN / "suppliers.csv", index=False)
suppliers

,category,supplier_id,supplier_name,lead_time_days,min_order_qty
0,Bathroom,SUP-BATH,Demo Bath Supply,5,10
1,Kitchen,SUP-KIT,Demo Kitchenware,7,6
2,Storage,SUP-STO,Demo Storage Co.,7,12
3,Bedding,SUP-BED,Demo Textile,10,8


## 6. คำนวณยอดขายต่อวัน และพอขายอีกกี่วัน

- ใช้ยอดขาย **30 วันล่าสุด** (1–30 ก.ค. 2026) เพราะสะท้อนความต้องการตอนนี้ได้ดีกว่าค่าเฉลี่ย 3 เดือน
- ขายต่อวัน = จำนวนชิ้นที่ขายใน 30 วัน ÷ 30
- พอขายอีกกี่วัน = สต็อกล่าสุด ÷ ขายต่อวัน
- ไม่นับแถวที่วันที่กำกวม (ถูกกักไว้ในข้อ 2)

In [8]:
WINDOW_START, WINDOW_END, WINDOW_DAYS = pd.Timestamp("2026-07-01"), pd.Timestamp("2026-07-30"), 30

lines = order_lines.merge(orders_clean[["order_id", "order_date_clean", "date_status"]], on="order_id")
in_window = lines[lines.order_date_clean.between(WINDOW_START, WINDOW_END)]
sales = in_window.groupby("product_id").agg(units_30d=("quantity", "sum"), orders_30d=("order_id", "nunique"))

risk = (products_clean[["product_id", "product_name", "category"]]
        .merge(stock_latest, on="product_id")
        .merge(sales, on="product_id", how="left")
        .merge(suppliers, on="category", how="left"))
risk[["units_30d", "orders_30d"]] = risk[["units_30d", "orders_30d"]].fillna(0).astype(int)
risk["daily_sales"] = (risk.units_30d / WINDOW_DAYS).round(2)
risk["days_remaining"] = np.where(risk.units_30d > 0, risk.available_qty / (risk.units_30d / WINDOW_DAYS), np.nan)
risk["days_remaining"] = risk.days_remaining.round(1)
risk[["product_id", "available_qty", "units_30d", "orders_30d", "daily_sales", "days_remaining", "lead_time_days"]].sort_values("days_remaining").head(10)

,product_id,available_qty,units_30d,orders_30d,daily_sales,days_remaining,lead_time_days
47,SKU-0048,0,7,5,0.23,0.0,10
4,SKU-0005,1,21,10,0.70,1.4,5
24,SKU-0025,1,14,6,0.47,2.1,7
31,SKU-0032,2,16,8,0.53,3.8,7
52,SKU-0053,3,16,7,0.53,5.6,7
0,SKU-0001,5,18,9,0.60,8.3,5
67,SKU-0068,2,7,3,0.23,8.6,7
17,SKU-0018,3,10,6,0.33,9.0,5
39,SKU-0040,5,15,6,0.50,10.0,10
13,SKU-0014,7,20,9,0.67,10.5,5


## 7. ตัดสินสถานะ

กฎเรียงตามลำดับ (เจอข้อไหนก่อนใช้ข้อนั้น):
1. **ต้องให้คนตรวจ** ถ้าไม่มีข้อมูลผู้ขาย หรือมีออร์เดอร์ใน 30 วันน้อยกว่า 3 รายการ (ยอดขายต่อวันจากข้อมูลน้อยขนาดนี้ไม่น่าเชื่อถือ)
2. **เสี่ยง** ถ้าสต็อกเป็น 0 หรือ พอขายได้น้อยกว่า เวลาส่งของ + 3 วันเผื่อ
3. **ปกติ** นอกเหนือจากนั้น

ถ้าสินค้ามีออร์เดอร์ที่ถูกกักไว้ จะเขียนหมายเหตุบอกด้วย แม้ไม่เปลี่ยนผล

In [9]:
SAFETY_DAYS, MIN_ORDERS = 3, 3
quarantined_skus = set(quarantine.product_id)

def decide(r):
    notes = []
    if r.product_id in quarantined_skus:
        notes.append("มีออร์เดอร์วันที่กำกวมที่ไม่ได้นับ (รอคนตรวจ)")
    if r.stock_source != "snapshot 06:00":
        notes.append(f"ใช้สต็อกอัปเดตล่าสุด ({r.stock_source})")
    if pd.isna(r.lead_time_days):
        return "ต้องให้คนตรวจ", "; ".join(["ไม่มีข้อมูลผู้ขาย"] + notes)
    if r.orders_30d < MIN_ORDERS:
        return "ต้องให้คนตรวจ", "; ".join([f"มีออร์เดอร์ใน 30 วันเพียง {r.orders_30d} รายการ ประเมินยอดขายไม่ได้"] + notes)
    if r.available_qty == 0:
        return "เสี่ยง", "; ".join(["สต็อกหมดแล้ว"] + notes)
    if r.days_remaining < r.lead_time_days + SAFETY_DAYS:
        return "เสี่ยง", "; ".join([f"พอขาย {r.days_remaining:g} วัน แต่ผู้ขายใช้ {r.lead_time_days:g} วัน (+{SAFETY_DAYS} วันเผื่อ)"] + notes)
    return "ปกติ", "; ".join(notes)

risk[["risk_status", "note"]] = risk.apply(lambda r: pd.Series(decide(r)), axis=1)
risk.risk_status.value_counts()

risk_status
ปกติ             70
เสี่ยง            7
ต้องให้คนตรวจ     5
Name: count, dtype: int64

In [10]:
risk[risk.risk_status != "ปกติ"][["product_id", "product_name", "available_qty", "daily_sales", "days_remaining",
                                  "lead_time_days", "risk_status", "note"]].sort_values("risk_status")

,product_id,product_name,available_qty,daily_sales,days_remaining,lead_time_days,risk_status,note
26,SKU-0027,BaanDee Item 27,31,0.10,310.0,5,ต้องให้คนตรวจ,มีออร์เดอร์ใน 30 วันเพียง 2 รายการ ประเมินยอดข...
30,SKU-0031,BaanDee Item 31,26,0.10,260.0,7,ต้องให้คนตรวจ,มีออร์เดอร์ใน 30 วันเพียง 2 รายการ ประเมินยอดข...
72,SKU-0073,BaanDee Item 73,8,0.10,80.0,10,ต้องให้คนตรวจ,มีออร์เดอร์ใน 30 วันเพียง 2 รายการ ประเมินยอดข...
80,SKU-KT-104,Compact Rice Cooker,3,0.03,90.0,7,ต้องให้คนตรวจ,มีออร์เดอร์ใน 30 วันเพียง 1 รายการ ประเมินยอดข...
81,SKU-BD-220,Cooling Cotton Sheet Set,12,0.03,360.0,10,ต้องให้คนตรวจ,มีออร์เดอร์ใน 30 วันเพียง 1 รายการ ประเมินยอดข...
4,SKU-0005,BaanDee Item 5,1,0.70,1.4,5,เสี่ยง,พอขาย 1.4 วัน แต่ผู้ขายใช้ 5 วัน (+3 วันเผื่อ)...
24,SKU-0025,BaanDee Item 25,1,0.47,2.1,7,เสี่ยง,พอขาย 2.1 วัน แต่ผู้ขายใช้ 7 วัน (+3 วันเผื่อ)
31,SKU-0032,BaanDee Item 32,2,0.53,3.8,7,เสี่ยง,พอขาย 3.8 วัน แต่ผู้ขายใช้ 7 วัน (+3 วันเผื่อ)
39,SKU-0040,BaanDee Item 40,5,0.50,10.0,10,เสี่ยง,พอขาย 10 วัน แต่ผู้ขายใช้ 10 วัน (+3 วันเผื่อ)
47,SKU-0048,BaanDee Item 48,0,0.23,0.0,10,เสี่ยง,สต็อกหมดแล้ว


## 8. บันทึกผลลัพธ์

ชื่อคอลัมน์ใน `stock_risk.csv` ตรงกับสัญญาข้อมูลระหว่าง n8n กับหน้าเว็บ

In [11]:
out = risk.rename(columns={"available_qty": "current_stock"})
out["stock_as_of"] = out.as_of.dt.strftime("%Y-%m-%d %H:%M")
out = out[["product_id", "product_name", "category", "current_stock", "stock_as_of", "stock_source",
           "daily_sales", "days_remaining", "lead_time_days", "supplier_id", "min_order_qty", "risk_status", "note"]]
out.to_csv(CLEAN / "stock_risk.csv", index=False)
pd.DataFrame(quality_log).to_csv(CLEAN / "data_quality_log.csv", index=False)
print("บันทึก:", [p.name for p in sorted(CLEAN.glob("*.csv"))], [p.name for p in GEN.glob("*.csv")])
pd.DataFrame(quality_log)

บันทึก: ['data_quality_log.csv', 'quarantine_order_dates.csv', 'stock_risk.csv'] ['suppliers.csv']


,issue,rows,action,evidence
0,วันที่รูปแบบ วัน/เดือน/ปี ที่ตีความได้แบบเดียว,1,แปลงเป็น ปี-เดือน-วัน,ส่วนแรกของวันที่มากกว่า 12 จึงเป็นวันแน่นอน
1,วันที่กำกวม (07/08/2026),1,ไม่นับในยอดขาย กักไว้ให้คนตรวจ (quarantine_ord...,อ่านได้ 2 แบบ และไม่ตรงกับวันที่ใน order_id
2,"ชื่อหมวดสินค้าไม่สม่ำเสมอ (ตัวพิมพ์เล็ก, 'Bed ...",9,แปลงเป็น 4 หมวดมาตรฐานตาม CATEGORY_MAP,SKU-0019: storage; SKU-0023: Bed room; SKU-003...
3,สต็อกสองแหล่งเวลาไม่ตรงกัน,2,ใช้ค่าล่าสุด และเก็บ stock_as_of กับ stock_source,SKU-KT-104: 8→3; SKU-BD-220: 0→12


## สรุปข้อจำกัด
- เวลาส่งของของผู้ขายเป็นสมมติฐานที่ทีมสร้าง ถ้าบริษัทมีข้อมูลจริงต้องเปลี่ยน
- ใช้ค่าเฉลี่ย 30 วัน ไม่ได้คิดฤดูกาลหรือโปรโมชัน
- สต็อกเป็นของคลังเดียว (BKK-DC) ณ วันที่ 31 ก.ค. 2026